In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import open3d as o3d
import glob

ROOT = "data/rpi_custom_dataset_01/scene_1"

# vizualization testing

In [ ]:
data = np.loadtxt(os.path.join(ROOT, "scene_1.txt"), delimiter=",")

In [ ]:
data.shape

In [ ]:
points = data[:, :3]
rgb = data[:, 3:6] / 255.0

In [ ]:
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)
pcd.colors = o3d.utility.Vector3dVector(rgb)

In [ ]:
viz = o3d.visualization.Visualizer()
viz.create_window()
viz.add_geometry(pcd)
viz.run()
viz.destroy_window()

___

In [ ]:
data = np.loadtxt(os.path.join(ROOT, "Coupling_4073350.txt"), delimiter=",")

In [ ]:
points = data[:, :3]
rgb = data[:, 3:6] / 255.0

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)
pcd.colors = o3d.utility.Vector3dVector(rgb)

viz = o3d.visualization.Visualizer()
viz.create_window()
viz.add_geometry(pcd)
viz.run()
viz.destroy_window()

getting the class names

In [ ]:
anno_dir = os.path.join(ROOT, "Annotations")

In [ ]:
len(os.listdir(anno_dir))

In [ ]:
n = os.listdir(anno_dir)[0]
n

In [ ]:
n.split("_")[0]

In [ ]:
n = set(map(lambda x: x.rsplit("_", 1)[0], os.listdir(anno_dir)))

In [ ]:
classes = list(n)
class2labels = {cls: i for i,cls in enumerate(classes)}

In [ ]:
classes

In [ ]:
len(classes)

In [ ]:
with open(r"data\rpi_custom_dataset_01_raw\classes.txt", "w") as f:
    for c in classes:
        f.write(f"{c}\n")

In [ ]:
class2labels

In [ ]:
points_list = []

In [ ]:
for f in glob.glob(os.path.join(anno_dir, "*.txt")):
    cls = os.path.basename(f).rsplit("_", 1)[0]
    print(f)
    if cls not in classes:
        cls = 'clutter'

    points = np.loadtxt(f, delimiter=",")
    if points.ndim == 1:
        points = points.reshape(1, -1)
    labels = np.ones((points.shape[0],1)) * class2labels[cls]
    points_list.append(np.concatenate([points, labels], 1))



In [ ]:
data_label = np.concatenate(points_list, 0)
xyz_min = np.amin(data_label, axis=0)[0:3]
data_label[:, 0:3] -= xyz_min

In [ ]:
data_label.shape

In [ ]:
import colorsys

In [ ]:
# data = np.load("data/rpi_custom_dataset_01/scene_1.npy")
data = np.load("data/stanford_indoor3d/Area_1_conferenceRoom_1.npy")

labels = data[:, 6].astype(int)

In [ ]:
labels

In [ ]:
CLASS_NAMES = ['Elbow',
 'Stairs',
 'Mechanical_Equipment',
 'Conduit_Elbow',
 'Duct',
 'HSS_Channel',
 'Wall',
 'Electrical_Equipment',
 'Conduit',
 'Light',
 'Reducer',
 'Valve',
 'Pipe',
 'Transition',
 'Floor',
 'Receptacle',
 'Tee',
 'Pressure_Gauge',
 'Mullion',
 'Coupling',
 'C_Channel']

In [ ]:
CLASS_COLORS = []
n = len(CLASS_NAMES)
for i, name in enumerate(CLASS_NAMES):
    hue = i / n
    rgb = colorsys.hsv_to_rgb(hue, 0.9, 0.9)
    CLASS_COLORS.append([c for c in rgb])

In [ ]:
CLASS_COLORS[labels]

In [ ]:
n = np.load("data\stanford_indoor3d\Area_1_conferenceRoom_1.npy")

In [ ]:
n.shape

In [ ]:
n[0]

In [ ]:
n

# some model tests

In [ ]:
import datetime
import importlib
import logging
import os
import shutil
import sys
import time
from pathlib import Path

import numpy as np
import torch
from tqdm import tqdm

import provider
from data_utils.S3DISDataLoader import S3DISDataset

from models import pointnet2_sem_seg

In [ ]:
root = "data/stanford_indoor3d"
NUM_CLASSES = 13
NUM_POINT = 4096
BATCH_SIZE = 32

In [ ]:
dataset = S3DISDataset(
    split="train",
    data_root=root,
    num_point=NUM_POINT,
    test_area=5,
    block_size=1.0,
    sample_rate=1.0,
    transform=None
)

In [ ]:
dataset[0][0].shape

In [ ]:
classifier = pointnet2_sem_seg.get_model(NUM_CLASSES).cuda()
classifier = classifier.eval()

In [ ]:
len(dataset)

In [ ]:
points, labels = dataset[0]

In [ ]:
points = torch.Tensor(points).float().cuda()
labels = torch.Tensor(labels).long().cuda()

In [ ]:
# add batch dim
points = points.unsqueeze(0)
labels = labels.unsqueeze(0)

In [ ]:
points = points.transpose(2, 1)

In [ ]:
seg_pred, _ = classifier(points)

In [ ]:
seg_pred = seg_pred.contiguous().view(-1, NUM_CLASSES)

In [ ]:
seg_pred.shape

In [ ]:
pred_val = seg_pred.contiguous().cpu().data.numpy()
pred_val = np.argmax(pred_val, 1)

In [ ]:
pred_val

In [ ]:
labels.shape

In [ ]:
labels

In [ ]:
points, _ = dataset[0]
points.shape

In [ ]:
p = np.column_stack(( points, pred_val ))
p.shape

In [ ]:
p

In [ ]:
######################

In [ ]:
points, target = dataset[0]
points, target = torch.Tensor(points).float().cuda().unsqueeze(0), torch.Tensor(target).float().cuda().unsqueeze(0)
points = points.transpose(2, 1)
points.shape, target.shape

In [ ]:
points = points.squeeze(0).cpu().numpy().transpose(1, 0)
target = target.squeeze(0).cpu().numpy()
points.shape, target.shape

In [ ]:
points = points[:, 0:6]
points.shape

In [ ]:
import open3d as o3d

In [ ]:
def make_pcd(data, color_mode="rgb"):
    pc = data[:, :3]
    rgb = data[:, 3:6]
    # labels = data[:, 6].astype(int)

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pc)
    # pcd.colors = o3d.utility.Vector3dVector(
    #     CLASS_COLORS[labels] if color_mode == "label" else rgb
    # )
    pcd.colors = o3d.utility.Vector3dVector(rgb)
    return pcd, points

pcd, _ = make_pcd(points)

In [ ]:
viz = o3d.visualization.Visualizer()
viz.create_window()
viz.add_geometry(pcd)
viz.run()
viz.destroy_window()

# dataset testing

In [ ]:
from data_utils.RPIDataLoader import PointNetDataset

In [ ]:
data_root = "data/rpi_data"
num_point, block_size, sample_rate, num_classes = 4096, 1.0, 0.01, 21

point_data = PointNetDataset(
    split="train",
    data_root=data_root,
    num_point=num_point,
    num_classes=num_classes,
    block_size=block_size,
    sample_rate=sample_rate,
    transform=None,
)

In [ ]:
import torch
from torch.utils.data import DataLoader
import numpy as np

In [ ]:
trainDataLoader = torch.utils.data.DataLoader(
        point_data,
        batch_size=32,
        shuffle=True,
        num_workers=1,
        pin_memory=True,
        # drop_last=True,
        persistent_workers=True,
        worker_init_fn=lambda x: np.random.seed(x + int(time.time())),
    )

len(trainDataLoader)

In [ ]:
len(trainDataLoader)

In [ ]:
print("point data size:", point_data.__len__())
print("point data 0 shape:", point_data.__getitem__(0)[0].shape)
print("point label 0 shape:", point_data.__getitem__(0)[1].shape)

In [ ]:
point_data[0][1].shape

In [ ]:
class DatasetWholeScene:
    # prepare to give prediction on each points
    def __init__(
        self,
        root,
        block_points=4096,
        stride=0.5,
        block_size=1.0,
        padding=0.001,
        num_classes=21,
    ):
        self.block_points = block_points
        self.block_size = block_size
        self.padding = padding
        self.root = root
        self.stride = stride
        self.scene_points_num = []
        self.file_list = [
            d for d in os.listdir(root)
        ]
        self.scene_points_list = []
        self.semantic_labels_list = []
        self.room_coord_min, self.room_coord_max = [], []
        for file in self.file_list:
            data = np.load(os.path.join(root, file))
            points = data[:, :3]
            self.scene_points_list.append(data[:, :6])
            self.semantic_labels_list.append(data[:, 6])
            coord_min, coord_max = (
                np.amin(points, axis=0)[:3],
                np.amax(points, axis=0)[:3],
            )
            # self.room_coord_min.append(coord_min), self.room_coord_max.append(coord_max)
        assert len(self.scene_points_list) == len(self.semantic_labels_list)

        labelweights = np.zeros(num_classes)
        for seg in self.semantic_labels_list:
            tmp, _ = np.histogram(seg, range(num_classes + 1))
            self.scene_points_num.append(seg.shape[0])
            labelweights += tmp
        labelweights = labelweights.astype(np.float32)
        labelweights = labelweights / np.sum(labelweights)
        self.labelweights = np.power(np.amax(labelweights) / labelweights, 1 / 3.0)

    def __getitem__(self, index):
        point_set_ini = self.scene_points_list[index]
        points = point_set_ini[:, :6]
        labels = self.semantic_labels_list[index]
        coord_min, coord_max = np.amin(points, axis=0)[:3], np.amax(points, axis=0)[:3]
        grid_x = int(
            np.ceil(float(coord_max[0] - coord_min[0] - self.block_size) / self.stride)
            + 1
        )
        grid_y = int(
            np.ceil(float(coord_max[1] - coord_min[1] - self.block_size) / self.stride)
            + 1
        )
        data_room, label_room, sample_weight, index_room = (
            np.array([]),
            np.array([]),
            np.array([]),
            np.array([]),
        )
        for index_y in range(0, grid_y):
            for index_x in range(0, grid_x):
                s_x = coord_min[0] + index_x * self.stride
                e_x = min(s_x + self.block_size, coord_max[0])
                s_x = e_x - self.block_size
                s_y = coord_min[1] + index_y * self.stride
                e_y = min(s_y + self.block_size, coord_max[1])
                s_y = e_y - self.block_size
                point_idxs = np.where(
                    (points[:, 0] >= s_x - self.padding)
                    & (points[:, 0] <= e_x + self.padding)
                    & (points[:, 1] >= s_y - self.padding)
                    & (points[:, 1] <= e_y + self.padding)
                )[0]
                if point_idxs.size == 0:
                    continue
                num_batch = int(np.ceil(point_idxs.size / self.block_points))
                point_size = int(num_batch * self.block_points)
                replace = (
                    False if (point_size - point_idxs.size <= point_idxs.size) else True
                )
                point_idxs_repeat = np.random.choice(
                    point_idxs, point_size - point_idxs.size, replace=replace
                )
                point_idxs = np.concatenate((point_idxs, point_idxs_repeat))
                np.random.shuffle(point_idxs)
                data_batch = points[point_idxs, :]
                normlized_xyz = np.zeros((point_size, 3))
                normlized_xyz[:, 0] = data_batch[:, 0] / coord_max[0]
                normlized_xyz[:, 1] = data_batch[:, 1] / coord_max[1]
                normlized_xyz[:, 2] = data_batch[:, 2] / coord_max[2]
                data_batch[:, 0] = data_batch[:, 0] - (s_x + self.block_size / 2.0)
                data_batch[:, 1] = data_batch[:, 1] - (s_y + self.block_size / 2.0)
                data_batch[:, 3:6] /= 255.0
                data_batch = np.concatenate((data_batch, normlized_xyz), axis=1)
                label_batch = labels[point_idxs].astype(int)
                batch_weight = self.labelweights[label_batch]

                data_room = (
                    np.vstack([data_room, data_batch]) if data_room.size else data_batch
                )
                label_room = (
                    np.hstack([label_room, label_batch])
                    if label_room.size
                    else label_batch
                )
                sample_weight = (
                    np.hstack([sample_weight, batch_weight])
                    if label_room.size
                    else batch_weight
                )
                index_room = (
                    np.hstack([index_room, point_idxs])
                    if index_room.size
                    else point_idxs
                )
        data_room = data_room.reshape((-1, self.block_points, data_room.shape[1]))
        label_room = label_room.reshape((-1, self.block_points))
        sample_weight = sample_weight.reshape((-1, self.block_points))
        index_room = index_room.reshape((-1, self.block_points))
        return data_room, label_room, sample_weight, index_room

    def __len__(self):
        return len(self.scene_points_list)



In [ ]:
data = DatasetWholeScene(root="data/rpi_data", block_points=4096, stride=0.5, block_size=1.0, padding=0.001, num_classes=21)

In [ ]:
data_room, label_room, sample_weight, index_room = data[0]

In [ ]:
data.file_list

In [ ]:
data_room.shape, label_room.shape, sample_weight.shape, index_room.shape

In [ ]:
whole_data_scene = data.scene_points_list[0]
whole_data_scene.shape

In [ ]:
whole_data_label = data.semantic_labels_list[0]
whole_data_label.shape

In [ ]:
BATCH_SIZE = 32

In [ ]:
num_blocks = data_room.shape[0]
s_batch_num = (num_blocks + BATCH_SIZE - 1) // BATCH_SIZE

In [ ]:
sbatch = 0
start_idx = sbatch * BATCH_SIZE
end_idx = min((sbatch + 1) * BATCH_SIZE, num_blocks)
real_batch_size = end_idx - start_idx

In [ ]:
batch_data = data_room[start_idx:end_idx]
batch_label = label_room[start_idx:end_idx]
batch_data.shape, batch_label.shape

In [ ]:
from models import pointnet2_sem_seg

In [ ]:
NUM_CLASSES = 21
classifier = pointnet2_sem_seg.get_model(NUM_CLASSES).cuda()

In [ ]:
torch_data = torch.Tensor(batch_data)
torch_data = torch_data.float().cuda()
torch_data = torch_data.transpose(2, 1)
seg_pred, _ = classifier(torch_data)
batch_pred_label = seg_pred.contiguous().cpu().data.max(2)[1].numpy()

# inference pipeline test

In [ ]:
import argparse
import os
from data_utils.RPIDataLoader import DatasetWholeScene
from data_utils.indoor3d_util import g_label2color
import torch
import logging
from pathlib import Path
import sys
import importlib
from tqdm import tqdm
import provider
import numpy as np
from models import pointnet2_sem_seg



In [ ]:
NUM_CLASSES = 21

In [ ]:
model = pointnet2_sem_seg.get_model(NUM_CLASSES).cuda()

In [ ]:
data = np.load("data/rpi_data/train_scene_1.npy")

In [ ]:
data.shape

In [ ]:
points = data[:, :6]
labels = data[:, 6].astype(int)
points.shape, labels.shape

In [ ]:
from RPIDataLoader import DatasetWholeScene

In [ ]:
ds = DatasetWholeScene(root="data/rpi_data", block_points=4096, stride=0.5, block_size=1.0, padding=0.001, num_classes=21)

In [ ]:
points = ds[0]

# more testing 

In [ ]:
import os
import numpy as np
import glob
import colorsys

In [ ]:
ROOT = "rpi_data_raw"

In [ ]:
classes = []
scenes = os.listdir(ROOT)
for scene in scenes:
    anno_dir = os.path.join(ROOT,scene, "_PointOut")
    anno = os.listdir(anno_dir)
    # print(f"Scene: {scene}, Number of Annotations: {len(anno)}")
    for a in anno:
        if ".xyz" in a:
            cls = a.rsplit("_", 1)[0]
            if cls not in classes:
                classes.append(cls)
classes
    

In [ ]:
if not os.path.exists(ROOT):
    print("dne  ")

In [ ]:
jos.mkdir()

In [ ]:
l = ["hi", "my", "name", "is", "jasper"]
with open("test.txt", "w") as file:
    for n in l:
        file.write(n + "\n")

In [ ]:
os.listdir("zips")

In [ ]:
os.path.isdir("zips.txt")

# more debugging

In [1]:
import numpy as np
from pathlib import Path
import os
from torch.utils.data import DataLoader
from tqdm import tqdm
import torch
import time

In [2]:
data_root = "rpi_data"

In [ ]:
dataset_path = Path(data_root)
unique_classes = set()

npy_files = list(dataset_path.glob('*.npy'))

print(f"Processing {len(npy_files)} files...")

for file_path in npy_files:
    data = np.load(file_path)
    
    if data.ndim == 2 and data.shape[1] == 7:
        classes_in_file = np.unique(data[:, 6])
        unique_classes.update(classes_in_file)
    else:
        print(f"Warning: Skipping {file_path.name} due to unexpected shape {data.shape}")

print("-" * 30)
print(f"Total unique classes found: {len(unique_classes)}")
print(f"Class IDs: {sorted(list(unique_classes))}")
print(f"Number of IDs: {len(unique_classes)}")

In [40]:
from models.pointnet2_sem_seg import get_model, get_loss
from data_utils.RPIDataLoader import PointNetDataset

In [30]:
NUM_CLASSES = 48
model = get_model(NUM_CLASSES).cuda()

In [43]:
ds = PointNetDataset(
        split="test",
        data_root=data_root,
        num_point=2048,
        num_classes=NUM_CLASSES,
        block_size=1.0,
        sample_rate=1.0,
        transform=None,
    )

Loading data from rpi_data for test set
Params: num_point=2048, block_size=1.0, sample_rate=1.0


100%|██████████| 1/1 [00:04<00:00,  4.21s/it]
c:\Users\jaspe\@projects\Pointnet_Pointnet2_pytorch\data_utils\RPIDataLoader.py:70: RuntimeWarning: divide by zero encountered in divide
  np.power(np.amax(labelweights) / labelweights, 1 / 3.0)
100%|██████████| 1/1 [00:03<00:00,  3.81s/it]

Totally 18788 samples in test set.


In [44]:
import open3d as o3d
import colorsys

with open("classes.txt", "r") as f:
    CLASS_NAMES = [line.strip() for line in f.readlines()]

CLASS_COLORS = []
n = len(CLASS_NAMES)
for i, name in enumerate(CLASS_NAMES):
    hue = i / n
    rgb = colorsys.hsv_to_rgb(hue, 0.9, 0.9)
    CLASS_COLORS.append([c for c in rgb])
CLASS_COLORS = np.array(CLASS_COLORS)

def visualize_point_cloud(points, targets, show_labels=True):
    xyz = points[:, :3]
    rgb = points[:, 3:6]
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(xyz)
    if show_labels:
        pcd.colors = o3d.utility.Vector3dVector(CLASS_COLORS[targets.astype(int)])
    else:
        pcd.colors = o3d.utility.Vector3dVector(rgb)
    o3d.visualization.draw_geometries([pcd])

In [45]:
points, target = ds[0]
points.shape, target.shape

((2048, 9), (2048,))

In [21]:
for i in range(10):
    points, target = ds[i]
    visualize_point_cloud(points, target, show_labels=False)

In [46]:
dataloader = DataLoader(
    ds,
    batch_size = 16,
    shuffle = True,
    num_workers = 0,
    # persistent_workers = True,
    # pin_memory = True
)

In [47]:
x, y = next(iter(dataloader))

In [48]:
x, y = x.float().cuda(), y.long().cuda()

In [49]:
out, trans_feat  = model(x.transpose(2, 1))

In [50]:
out.shape, y.shape

(torch.Size([16, 2048, 48]), torch.Size([16, 2048]))

In [51]:
out = out.contiguous().view(-1, NUM_CLASSES)
batch_label = y.view(-1,1)[:, 0].cpu().data.numpy()
y = y.view(-1,1)[:, 0]

In [55]:
weights = torch.Tensor(ds.labelweights).cuda()
criterion = get_loss().cuda()

In [56]:
loss = criterion(out, y, trans_feat, weights)

In [57]:
loss.item()

4.155230522155762

In [58]:
loss.backward()

In [60]:
for i, (points, target) in enumerate(dataloader):
    points, target = points.float().cuda(), target.long().cuda()
    points = points.transpose(2,1)
    out, _ = model(points)
    out = out.contiguous().view(-1, NUM_CLASSES)
    batch_label = target.view(-1,1)[:, 0].cpu().data.numpy()
    target = target.view(-1,1)[:, 0]
    loss = criterion(out, target, None, weights)
    print(f"{i} - Loss: {loss.item():.4f}")

0 - Loss: 4.1442
1 - Loss: 4.1405
2 - Loss: 4.1194
3 - Loss: 4.1286
4 - Loss: 4.1278
5 - Loss: 4.1350
6 - Loss: 4.0942
7 - Loss: 4.1626
8 - Loss: 4.1187
9 - Loss: 4.0869
10 - Loss: 4.1525
11 - Loss: 4.1248
12 - Loss: 4.1691
13 - Loss: 4.1418
14 - Loss: 4.1550


KeyboardInterrupt: 